# Pollution along the growth path, with NO₂

Step 2 of the pollution follow-up (`docs/successors/pollution-no2-prespec.md`, confirmed 2026-10-30): surface NO₂ (Anenberg et al. 2022) as a second outcome, on DOSE output (main arm, 2005–2018) and on night lights per person (check arm, 2005–2012). Descriptive only.

This notebook first runs the coverage gate ◇ G3. **G3 aggregates no NO₂ value:** it counts each region's valid 1 km NO₂ land cells (non-missing, at most 100 ppb).

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('../..'))
os.chdir(os.path.abspath('../..'))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import src.analysis.pollution_no2 as pn
counts = pn.region_no2_counts()
print(counts.GID_1.nunique(), 'regions,', counts.year.min(), '-', counts.year.max())
cells = pn.build_no2_counts()
print('1 km land cells above 100 ppb (set missing), by year:', cells.groupby('year').n_high.sum().to_dict())
print('valid 1 km land cells, by year (millions):', (cells.groupby('year').n_valid.sum() / 1e6).round(1).to_dict())

3274 regions, 2005 - 2020


1 km land cells above 100 ppb (set missing), by year: {2005: 374, 2006: 407, 2007: 385, 2008: 353, 2009: 332, 2010: 206, 2011: 0, 2012: 139, 2013: 290, 2014: 270, 2015: 349, 2016: 341, 2017: 323, 2018: 348, 2019: 334, 2020: 197}


valid 1 km land cells, by year (millions): {2005: 115.8, 2006: 115.8, 2007: 115.8, 2008: 115.8, 2009: 115.7, 2010: 115.8, 2011: 115.8, 2012: 115.8, 2013: 115.8, 2014: 115.8, 2015: 115.8, 2016: 115.8, 2017: 115.8, 2018: 115.8, 2019: 115.8, 2020: 115.8}


## ◇ G3, DOSE arm (main): idea B's sample, 2005–2018

In [2]:
dk, dd = pn.dose_arm(counts)
print('sample:', len(dk), 'region-years,', dk.GID_1.nunique(), 'regions,', dk.GID_0.nunique(), 'countries')
print('dropped by the NO2 cell rule (< 500 valid cells):', len(dd), 'region-years in', dd.GID_1.nunique(), 'regions;',
      'by country:', dd.groupby('GID_0').GID_1.nunique().sort_values(ascending=False).head(10).to_dict())
gd = pn.g3(dk)
print(gd['by_tercile'], '\n')
n = gd['regions'].copy(); n.columns = pn.P.SEGMENT_LABELS; print(n, '\n')
print('comparisons:', gd['comparisons'])
print('G3 passes (DOSE arm):', gd['passes'])

sample: 18645 region-years, 1493 regions, 75 countries
dropped by the NO2 cell rule (< 500 valid cells): 648 region-years in 51 regions; by country: {'NZL': 15, 'CHE': 7, 'KGZ': 2, 'KOR': 2, 'NOR': 2, 'MYS': 2, 'IND': 2, 'BEL': 1, 'ARG': 1, 'ARE': 1}


         countries  regions  passes
tercile                            
0.0             26      542    True
1.0             24      569    True
2.0             25      382    True 

         < 1,000  1,000-4,000  4,000-12,000  > 12,000
tercile                                              
0.0          116          392           210        40
1.0           34          258           317       147
2.0            0           10           117       311 

comparisons: [('seg1', 0, 1), ('seg2', 0, 1), ('seg2', 1, 2), ('seg3', 1, 2)]
G3 passes (DOSE arm): True


## ◇ G3, lights arm (check): step 1's sample, knots and terciles, 2005–2012

In [3]:
lk, ld, knots = pn.lights_arm(counts)
print('knots (step 1, reproduced):', knots.round(3))
print('sample:', len(lk), 'region-years,', lk.GID_1.nunique(), 'regions,', lk.GID_0.nunique(), 'countries')
print('dropped by the NO2 cell rule:', len(ld), 'region-years in', ld.GID_1.nunique(), 'regions;',
      'by country:', ld.groupby('GID_0').GID_1.nunique().sort_values(ascending=False).head(10).to_dict())
gl = pn.g3(lk)
print(gl['by_tercile'], '\n')
n = gl['regions'].copy(); n.columns = pn.P.SEGMENT_LABELS; print(n, '\n')
print('comparisons:', gl['comparisons'])
print('G3 passes (lights arm):', gl['passes'])
h = lk[(lk.tercile == 2) & (lk.segment == 0)]
print('high tercile below 1,000 USD, regions by country:', h.groupby('GID_0').GID_1.nunique().sort_values(ascending=False).to_dict())

knots (step 1, reproduced): [-4.022 -3.013 -2.213]
sample: 21653 region-years, 2743 regions, 161 countries
dropped by the NO2 cell rule: 481 region-years in 63 regions; by country: {'NZL': 7, 'CPV': 7, 'MUS': 5, 'TTO': 3, 'PHL': 3, 'MNE': 3, 'SLB': 2, 'BHR': 2, 'IND': 2, 'COM': 2}
         countries  regions  passes
tercile                            
0.0             55      802    True
1.0             53     1188    True
2.0             53      753    True 

         < 1,000  1,000-4,000  4,000-12,000  > 12,000
tercile                                              
0.0          460          208           159       142
1.0          590          442           423       199
2.0           66          140           292       440 

comparisons: [('seg0', 0, 1), ('seg0', 1, 2), ('seg1', 0, 1), ('seg1', 1, 2), ('seg2', 0, 1), ('seg2', 1, 2), ('seg3', 0, 1), ('seg3', 1, 2)]
G3 passes (lights arm): True
high tercile below 1,000 USD, regions by country: {'IND': 21, 'SEN': 14, 'MNG': 12, 'PAN': 5,